# Khám phá giao dịch và khai thác tập mục

Notebook này nhận gói .tar.gz do notebook 02 tạo, xác minh checksum và hợp đồng artifact, sau đó phân tích giao dịch trong gói. Mọi xử lý, đo lường, lưu bảng và tạo hình đều chạy bằng Julia.

**Mục lục**

1. Chọn gói bàn giao
2. Giải nén an toàn và đọc manifest
3. Kiểm định giao dịch, ánh xạ ảnh và nhãn metadata
4. Khám phá độ dài giao dịch và tần suất từ thị giác
5. Đối chiếu Eclat Set và BitVector
6. Đo thời gian và peak RSS
7. Phân tích mẫu và làm giàu theo lớp
8. Truy vết ảnh minh họa
9. Lưu và tải kết quả
10. Kết luận và giới hạn diễn giải


### Thiết lập Julia và source

Cell đầu tìm checkout đúng phiên bản; nếu không thấy, Colab tải commit đã ghim. Tiếp theo nạp project Julia và các hàm hiển thị dùng trong notebook.


In [ ]:
# Julia runtime: chạy trực tiếp bằng kernel Julia, không cần Python điều phối.
# Colab: chọn Runtime → Change runtime type → Julia và CPU trước khi chạy.
# JULIA_NUM_THREADS phải được đặt trước khi kernel khởi động; cell này chỉ ghi lại yêu cầu.
ENV["JULIA_NUM_THREADS"] = "1"
ENV["OPENBLAS_NUM_THREADS"] = "1"
ENV["OPENBLAS_CORETYPE"] = "Haswell"
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"

using Pkg, SHA, Markdown, LibGit2, Base64
using LinearAlgebra

REPOSITORY_URL = "https://github.com/NguyenVoTanDuy2006/DataMining-FIM-Project.git"
SOURCE_BRANCH = "feature/fashion-mnist-consolidated"
SOURCE_COMMIT = "dbadabc4af05070c602b3bcd890150fcd496c885"

function find_fashion_repo(start_path)
    path = realpath(start_path)
    while true
        if isfile(joinpath(path, "Project.toml")) && isfile(joinpath(path, "src", "EclatFIM.jl"))
            return path
        end
        parent = dirname(path)
        parent == path && return nothing
        path = parent
    end
end

local_candidate = find_fashion_repo(pwd())
local_commit = if isnothing(local_candidate)
    nothing
else
    repo = LibGit2.GitRepo(local_candidate)
    try
        string(LibGit2.head_oid(repo))
    finally
        close(repo)
    end
end
USING_LOCAL_REPO = local_commit == SOURCE_COMMIT
if USING_LOCAL_REPO
    REPO_ROOT = local_candidate
    actual_commit = local_commit
else
    REPO_ROOT = joinpath("/content", "fashion-fim-" * first(SOURCE_COMMIT, 8))
    git_repo = if !ispath(REPO_ROOT)
        mkpath(dirname(REPO_ROOT))
        LibGit2.clone(REPOSITORY_URL, REPO_ROOT; branch=SOURCE_BRANCH)
    elseif isdir(joinpath(REPO_ROOT, ".git"))
        LibGit2.GitRepo(REPO_ROOT)
    else
        error("$REPO_ROOT exists but is not the pinned Git repository. Restart the Colab Julia runtime.")
    end
    actual_commit = try
        LibGit2.checkout!(git_repo, SOURCE_COMMIT; force=true)
        string(LibGit2.head_oid(git_repo))
    finally
        close(git_repo)
    end
end
actual_commit == SOURCE_COMMIT || error("Expected source $SOURCE_COMMIT, found $actual_commit. Restart the runtime.")

if USING_LOCAL_REPO
    DEPOT_ROOT = first(DEPOT_PATH)
    DEMO_OUTPUT_ROOT = joinpath(tempdir(), "fashion-demo-output-" * first(SOURCE_COMMIT, 8))
    mkpath(DEMO_OUTPUT_ROOT)
else
    DEPOT_ROOT = joinpath("/content", "fashion-demo-work", "julia_depot")
    mkpath(DEPOT_ROOT)
    DEPOT_ROOT in DEPOT_PATH || pushfirst!(DEPOT_PATH, DEPOT_ROOT)
    DEMO_OUTPUT_ROOT = joinpath("/content", "fashion-demo-output")
    mkpath(DEMO_OUTPUT_ROOT)
end
ENV["JULIA_DEPOT_PATH"] = join(DEPOT_PATH, ":")
ENV["FASHION_DEMO_REPO_ROOT"] = REPO_ROOT
ENV["FASHION_DEMO_SOURCE_COMMIT"] = SOURCE_COMMIT
ENV["FASHION_DEMO_OUTPUT_ROOT"] = DEMO_OUTPUT_ROOT
ENV["FASHION_DEMO_LOCAL_REPO"] = USING_LOCAL_REPO ? "1" : "0"


#### Nạp project Julia và tiện ích hiển thị

Kích hoạt project đã ghim, nạp các module Julia, giới hạn BLAS ở một luồng và chuẩn bị hàm hiển thị Markdown, ảnh.

In [ ]:
Pkg.activate(REPO_ROOT)
Pkg.instantiate()
using EclatFIM
using EclatFIM.FashionMNIST: FashionTC, FashionFIMWorkflow
BLAS.set_num_threads(1)

function show_demo_markdown(path)
    display(Markdown.parse(read(path, String)))
end
function show_demo_image(path; mime="image/png", alt=basename(path))
    encoded = base64encode(read(path))
    safe_alt = replace(alt, "&" => "&amp;", "\"" => "&quot;", "<" => "&lt;", ">" => "&gt;")
    html = "<figure><img alt=\"$safe_alt\" src=\"data:$mime;base64,$encoded\" style=\"max-width:100%;height:auto\"><figcaption>$safe_alt</figcaption></figure>"
    display(MIME"text/html"(), html)
end

println("Source commit: ", actual_commit)
println("Execution mode: ", USING_LOCAL_REPO ? "local repository" : "Google Colab")
println("Julia runtime: ", VERSION)
println("Active Julia threads: ", Threads.nthreads(), " (fixed when the kernel started)")
println("OpenBLAS threads: ", BLAS.get_num_threads(), "; requested core target: ", ENV["OPENBLAS_CORETYPE"])
println("Julia depot: ", DEPOT_ROOT)
println("Demo outputs: ", DEMO_OUTPUT_ROOT)
println("EclatFIM and Fashion-MNIST are ready in this Julia kernel.")

## 1. Chọn gói bàn giao

Trong Colab, tải đúng một gói từ notebook 02 lên bằng Files panel. Khi chạy local, notebook tự tìm gói trong thư mục output của demo. Nếu có nhiều gói, đặt `FASHION_DEMO_BUNDLE_PATH` trước khi chạy hoặc chỉnh `BUNDLE_PATH` tại cell dưới.

In [ ]:
BUNDLE_PATH = get(ENV, "FASHION_DEMO_BUNDLE_PATH", "")
if isempty(BUNDLE_PATH)
    search_dirs = unique(filter(isdir, [
        joinpath(ENV["FASHION_DEMO_OUTPUT_ROOT"]), "/content"
    ]))
    archive_candidates = sort(unique(vcat([
        [joinpath(directory, name) for name in readdir(directory)
            if endswith(name, ".tar.gz") && isfile(joinpath(directory, name))]
        for directory in search_dirs
    ]...)))
    if isempty(archive_candidates)
        error("No .tar.gz found. Upload the notebook 02 bundle to /content or set FASHION_DEMO_BUNDLE_PATH.")
    elseif length(archive_candidates) > 1
        println.(archive_candidates)
        error("More than one demo bundle found. Set FASHION_DEMO_BUNDLE_PATH to the intended file.")
    end
    BUNDLE_PATH = only(archive_candidates)
end
isfile(BUNDLE_PATH) || error("Bundle file not found: $BUNDLE_PATH")
endswith(BUNDLE_PATH, ".tar.gz") || error("Select the .tar.gz package exported by notebook 02.")
EXTRACT_ROOT = mktempdir(prefix="fashion-demo-verified-")
VALIDATION_REPORT = joinpath(EXTRACT_ROOT, "validation_summary.md")
ENV["FASHION_DEMO_ARCHIVE"] = BUNDLE_PATH
ENV["FASHION_DEMO_EXTRACT_ROOT"] = EXTRACT_ROOT
println("Bundle: ", BUNDLE_PATH)
println("Private extraction directory: ", EXTRACT_ROOT)

## 2. Giải nén an toàn và đọc manifest

Trước khi giải nén, kiểm tra đường dẫn trong TAR, loại entry, kích thước gói và tổng dung lượng sau giải nén. Sau đó kiểm checksum ba manifest/metrics được gói demo tham chiếu.

In [ ]:
using EclatFIM
using EclatFIM.FashionMNIST: FASHION_FILES, FashionFIMWorkflow, read_dataset_manifest
using SHA, TOML, Tar, CodecZlib

archive_path = ENV["FASHION_DEMO_ARCHIVE"]
extract_root = ENV["FASHION_DEMO_EXTRACT_ROOT"]
repo_root = ENV["FASHION_DEMO_REPO_ROOT"]
source_commit = ENV["FASHION_DEMO_SOURCE_COMMIT"]

function safe_relative_path(value; allow_trailing=false)
    original = String(value)
    isempty(original) && error("Empty archive or manifest path.")
    occursin('\\', original) && error("Backslash path is rejected: $original")
    startswith(original, "/") && error("Absolute path is rejected: $original")
    occursin(r"^[A-Za-z]:", original) && error("Drive-qualified path is rejected: $original")
    normalized = allow_trailing ? replace(original, r"/+$" => "") : original
    isempty(normalized) && error("Empty normalized path.")
    parts = split(normalized, '/')
    any(part -> isempty(part) || part == "." || part == "..", parts) &&
        error("Non-canonical or traversing path is rejected: $original")
    return join(parts, '/')
end
file_sha(path) = bytes2hex(open(sha256, path))

isfile(archive_path) || error("Bundle file is missing: $archive_path")
filesize(archive_path) <= 300_000_000 || error("Bundle exceeds the 300 MB input limit.")

headers = open(archive_path, "r") do compressed
    stream = GzipDecompressorStream(compressed)
    try
        Tar.list(stream; strict=false)
    finally
        close(stream)
    end
end
member_paths = Set{String}()
expanded_bytes = 0
for header in headers
    path = safe_relative_path(header.path; allow_trailing=header.type == :directory)
    path in member_paths && error("Duplicate archive path: $path")
    push!(member_paths, path)
    header.type in (:file, :directory) || error("Only regular files and directories are allowed: $path ($(header.type)).")
    header.size >= 0 || error("Negative archive entry size: $path")
    expanded_bytes += header.size
    expanded_bytes <= 1_000_000_000 || error("Expanded bundle exceeds the 1 GB limit.")
end
"demo_manifest.toml" in member_paths || error("Missing file: demo_manifest.toml")
mkpath(extract_root)
open(archive_path, "r") do compressed
    stream = GzipDecompressorStream(compressed)
    try
        Tar.extract(stream, extract_root; set_permissions=false, copy_symlinks=false)
    finally
        close(stream)
    end
end

demo_path = joinpath(extract_root, "demo_manifest.toml")
demo = TOML.parsefile(demo_path)
demo["schema_version"] == 1 || error("Unsupported demo manifest schema version.")
demo["source_commit"] == source_commit || error("Bundle source commit differs from the pinned notebook source.")
demo["julia_version"] == string(VERSION) || error("Bundle Julia version does not match this session.")
run_id = String(demo["run_id"])
tc_run_id = String(demo["tc_run_id"])
occursin(r"^[A-Za-z0-9][A-Za-z0-9_-]*$", run_id) || error("Invalid dataset run ID.")
occursin(r"^[A-Za-z0-9][A-Za-z0-9_-]*$", tc_run_id) || error("Invalid transaction run ID.")

function checked_manifest_entry(bundle_root, entry, name)
    relative = safe_relative_path(entry["path"])
    path = joinpath(bundle_root, split(relative, '/')...)
    isfile(path) || error("Missing file referenced by $name: $relative")
    actual = file_sha(path)
    actual == lowercase(String(entry["sha256"])) || error("SHA-256 mismatch in $name: $relative")
    return (relative, path, actual)
end

dataset_relative, dataset_path, dataset_sha =
    checked_manifest_entry(extract_root, demo["dataset_manifest"], "dataset_manifest")
transaction_relative, transaction_manifest_path, transaction_manifest_sha =
    checked_manifest_entry(extract_root, demo["transaction_manifest"], "transaction_manifest")
fit_relative, fit_metrics_path, fit_metrics_sha =
    checked_manifest_entry(extract_root, demo["fit_metrics"], "fit_metrics")

dataset = TOML.parsefile(dataset_path)
transaction_manifest = TOML.parsefile(transaction_manifest_path)
fit_metrics = TOML.parsefile(fit_metrics_path)

## 3. Kiểm định giao dịch, ánh xạ ảnh và nhãn metadata

Các cell tiếp theo kiểm tra cấu hình K=32/seed 123, đủ tám artifact, checksum JSONL, TID liên tiếp từ 1, quy tắc split:index và việc nhãn chỉ xuất hiện trong metadata.

In [ ]:
dataset["run_id"] == run_id || error("Dataset manifest run ID differs from demo manifest.")
transaction_manifest["run_id"] == run_id || error("Transaction manifest run ID differs from demo manifest.")
transaction_manifest["tc_run_id"] == tc_run_id || error("Transaction run ID differs from demo manifest.")
dataset["validation_status"] == "PASS" || error("Dataset manifest did not pass validation.")
transaction_manifest["status"] == "PASS" || error("Transaction manifest did not pass validation.")
dataset["train_count"] == 60_000 && dataset["test_count"] == 10_000 ||
    error("Dataset manifest split counts differ from the release contract.")
transaction_manifest["image_count"] == 70_000 &&
    transaction_manifest["transaction_count"] == 70_000 &&
    transaction_manifest["support_denominator"] == 70_000 ||
    error("Transaction manifest counts differ from the 70,000-image contract.")
transaction_manifest["dataset_manifest_path"] == dataset_relative ||
    error("Transaction manifest references a different dataset manifest.")
transaction_manifest["dataset_manifest_sha256"] == dataset_sha ||
    error("Transaction manifest dataset SHA-256 does not match.")
get(transaction_manifest["configuration"], "K", 0) == 32 &&
    get(transaction_manifest["configuration"], "seed", 0) == 123 ||
    error("Transaction manifest does not describe the release K=32, seed=123 configuration.")
fit_metrics["run_id"] == tc_run_id || error("K-means metrics belong to a different transaction run.")
fit_metrics["k"] == 32 && fit_metrics["seed"] == 123 ||
    error("K-means metrics do not match K=32 and seed=123.")

### Kiểm tra artifact và checksum

Xác nhận gói có đúng tám artifact bắt buộc, đường dẫn hợp lệ, checksum khớp manifest và báo cáo toàn vẹn đạt `PASS`.

In [ ]:
required_artifacts = Dict(
    "transactions" => "transactions.jsonl",
    "codebook" => "codebook.bin",
    "centroids" => "centroids.csv",
    "patch_assignments" => "patch_assignments.csv",
    "tid_image_mapping" => "tid_image_mapping.csv",
    "image_metadata" => "image_metadata.csv",
    "config" => "config_snapshot.toml",
    "integrity_report" => "integrity_report.toml",
)
artifact_table = transaction_manifest["artifacts"]
Set(keys(artifact_table)) == Set(keys(required_artifacts)) ||
    error("Transaction manifest must contain exactly the eight release artifacts.")
artifact_paths = Dict{String,String}()
for (name, expected_filename) in required_artifacts
    entry = artifact_table[name]
    relative = safe_relative_path(entry["path"])
    endswith(relative, "/" * expected_filename) || error("Unexpected artifact path for $name: $relative")
    path = joinpath(extract_root, split(relative, '/')...)
    isfile(path) || error("Missing required artifact $name: $relative")
    file_sha(path) == lowercase(String(entry["sha256"])) || error("Artifact SHA-256 mismatch: $relative")
    artifact_paths[name] = path
end

lowercase(String(transaction_manifest["artifacts"]["transactions"]["sha256"])) ==
    lowercase(String(transaction_manifest["transaction_sha256"])) ||
    error("Transaction SHA-256 differs across the transaction manifest fields.")
integrity = TOML.parsefile(artifact_paths["integrity_report"])
integrity["status"] == "PASS" || error("The release integrity report did not pass.")

# Recheck that all official archive hashes recorded in the dataset manifest are recognized.
expected_archives = Dict(file.name => file.sha256 for file in FASHION_FILES)
Dict(String(k) => lowercase(String(v)) for (k,v) in dataset["archive_sha256"]) == expected_archives ||
    error("Dataset manifest contains unexpected official archive hashes.")

transactions_path = artifact_paths["transactions"]
input = FashionFIMWorkflow.load_transactions(transactions_path;
    expected_n=70_000, expected_items=1:32, require_nonempty=false, unique_mapping=true)
input.tid == collect(1:70_000) || error("Transaction IDs must be the ordered range 1 through 70,000.")
all(tx -> issorted(tx) && length(unique(tx)) == length(tx), input.transactions) ||
    error("A transaction contains unordered or duplicate visual words.")
input.input_sha256 == lowercase(String(transaction_manifest["transaction_sha256"])) ||
    error("Transaction JSONL does not match its recorded SHA-256.")
any(line -> occursin("\"label\"", line), eachline(transactions_path)) &&
    error("Class labels must not be embedded in the transaction JSONL.")

### Đối chiếu TID với ảnh và nhãn

Đọc bảng ánh xạ để xác minh 70.000 TID theo thứ tự, sau đó đối chiếu split, chỉ số ảnh và nhãn lớp trong metadata.

In [ ]:
image_for_tid = Vector{String}(undef, 70_000)
mapping_rows = open(artifact_paths["tid_image_mapping"], "r") do io
    readline(io) == "tid,image_id,split,source_index" || error("Unexpected image mapping header.")
    rows = 0
    for (row, line) in enumerate(eachline(io))
        rows += 1
        fields = split(line, ',')
        length(fields) == 4 || error("Malformed image mapping row $row.")
        tid = parse(Int, fields[1])
        tid == row || error("Image mapping TID is out of order at row $row.")
        split_name, source_index = fields[3], parse(Int, fields[4])
        expected_split, expected_index = row <= 60_000 ? ("train", row-1) : ("test", row-60_001)
        fields[2] == "$(expected_split):$(expected_index)" &&
            split_name == expected_split && source_index == expected_index ||
            error("Image mapping differs from the release order at TID $row.")
        image_for_tid[row] = fields[2]
        String(input.image_id[row]) == fields[2] || error("Transaction image ID differs from the TID mapping at row $row.")
        (split_name == "train" && 0 <= source_index < 60_000 ||
            split_name == "test" && 0 <= source_index < 10_000) ||
            error("Image source index is outside its split at TID $row.")
    end
    rows == 70_000 || error("Image mapping must contain exactly 70,000 rows.")
    rows
end

class_for_image = Dict{String,Int}()
open(artifact_paths["image_metadata"], "r") do io
    readline(io) == "image_id,split,source_index,class_label" || error("Unexpected image metadata header.")
    for (row, line) in enumerate(eachline(io))
        fields = split(line, ',')
        length(fields) == 4 || error("Malformed image metadata row $row.")
        image_id = fields[1]
        image_id == image_for_tid[row] || error("Metadata image does not match TID mapping at row $row.")
        expected_split, expected_index = row <= 60_000 ? ("train", row-1) : ("test", row-60_001)
        fields[2] == expected_split && parse(Int, fields[3]) == expected_index ||
            error("Metadata split/index differs from its image ID at row $row.")
        label = parse(Int, fields[4])
        0 <= label <= 9 || error("Class label is outside 0–9 at row $row.")
        class_for_image[image_id] = label
    end
    length(class_for_image) == 70_000 || error("Metadata must contain exactly 70,000 rows.")
end

### Hoàn tất kiểm định gói

Kiểm tra manifest giao dịch và manifest K-means cùng thuộc một lượt phát hành, rồi ghi tóm tắt kiểm định để hiển thị.

In [ ]:
dataset_run_root = safe_relative_path(dataset["run_root"])
startswith(dataset_run_root, "results/fashion/release/") ||
    error("Dataset manifest run root is outside the release output directory.")
expected_transaction_prefix = dataset_run_root * "/transactions/"
startswith(transaction_relative, expected_transaction_prefix) ||
    error("Transaction manifest is from a different release run.")
startswith(fit_relative, dataset_run_root * "/") || error("K-means metrics are from a different release run.")
open(joinpath(extract_root, "validation_summary.md"), "w") do io
    println(io, "## Gói đã xác minh")
    println(io)
    println(io, "- Mã lượt: ", run_id)
    println(io, "- Mã lượt từ điển: ", tc_run_id)
    println(io, "- Commit nguồn: ", demo["source_commit"])
    println(io, "- Trạng thái kiểm định: PASS")
    println(io, "- Số giao dịch: ", length(input.transactions))
    println(io, "- Giao dịch rỗng được giữ trong mẫu số: ", input.empty_count)
    println(io, "- SHA-256 giao dịch: ", input.input_sha256)
    println(io, "- Trạng thái K-means: ", fit_metrics["status"], "; hội tụ: ", fit_metrics["converged"],
        "; số vòng: ", fit_metrics["iterations"], "/", fit_metrics["max_iterations"])
end
println(joinpath(extract_root, "validation_summary.md"))
show_demo_markdown(VALIDATION_REPORT)

## 4. Khám phá độ dài giao dịch và tần suất từ thị giác

Giao dịch rỗng được giữ trong phân bố và trong mẫu số 70.000. Biểu đồ đầu cho biết số từ thị giác trên mỗi ảnh; biểu đồ thứ hai cho biết tần suất từng từ trong toàn bộ giao dịch.

In [ ]:
using EclatFIM
using EclatFIM.FashionMNIST: FASHION_CLASSES, FashionFIMWorkflow,
    FashionTransactionIO, download_fashion_mnist, load_fashion_mnist,
    parse_image_id, read_dataset_manifest
using SHA, TOML, PNGFiles
repo_root = ENV["FASHION_DEMO_REPO_ROOT"]
bundle_root = ENV["FASHION_DEMO_EXTRACT_ROOT"]
demo_manifest_path = joinpath(bundle_root, "demo_manifest.toml")
demo = TOML.parsefile(demo_manifest_path)
dataset_manifest_path = joinpath(bundle_root, split(demo["dataset_manifest"]["path"], '/')...)
transaction_manifest_path = joinpath(bundle_root, split(demo["transaction_manifest"]["path"], '/')...)
transaction_manifest = TOML.parsefile(transaction_manifest_path)
run_root = TOML.parsefile(dataset_manifest_path)["run_root"]

run_id = String(demo["run_id"])
results_root = get(ENV, "FASHION_DEMO_RESULTS_ROOT", "")
if isempty(results_root)
    results_root = isdir("/content") ? "/content" : joinpath(repo_root, "results", "fashion-demo-results")
end
mkpath(results_root)
folder_stem = "fashion-demo-results-" * replace(run_id, r"[^A-Za-z0-9._-]" => "_")
previous_results_dir = get(ENV, "FASHION_DEMO_ANALYSIS_DIR", "")
previous_run_id = get(ENV, "FASHION_DEMO_ANALYSIS_RUN_ID", "")
if previous_run_id == run_id && !isempty(previous_results_dir) && isdir(previous_results_dir)
    ANALYSIS_DIR = previous_results_dir
else
    ANALYSIS_DIR = joinpath(results_root, folder_stem)
    suffix = 2
    while ispath(ANALYSIS_DIR)
        ANALYSIS_DIR = joinpath(results_root, "$(folder_stem)-$(suffix)")
        suffix += 1
    end
end
mkpath(ANALYSIS_DIR)
ENV["FASHION_DEMO_ANALYSIS_DIR"] = ANALYSIS_DIR
ENV["FASHION_DEMO_ANALYSIS_RUN_ID"] = run_id
output_dir = ANALYSIS_DIR

cp(BUNDLE_PATH, joinpath(output_dir, "input_bundle.tar.gz"); force=true)
cp(demo_manifest_path, joinpath(output_dir, "demo_manifest.toml"); force=true)
cp(dataset_manifest_path, joinpath(output_dir, "dataset_manifest.toml"); force=true)
cp(transaction_manifest_path, joinpath(output_dir, "transaction_manifest.toml"); force=true)
isfile(VALIDATION_REPORT) && cp(VALIDATION_REPORT, joinpath(output_dir, "validation_summary.md"); force=true)
println("Persistent result folder: ", output_dir)
transaction_artifacts = transaction_manifest["artifacts"]
artifact_path(name) = joinpath(bundle_root, split(transaction_artifacts[name]["path"], '/')...)
transactions_path = artifact_path("transactions")
input = FashionFIMWorkflow.load_transactions(transactions_path;
    expected_n=70_000, expected_items=1:32, require_nonempty=false, unique_mapping=true)
transactions = input.transactions
n_transactions = length(transactions)

length_histogram = Dict{Int,Int}()
for transaction in transactions
    length_histogram[length(transaction)] = get(length_histogram, length(transaction), 0) + 1
end
transaction_lengths = [length(transaction) for transaction in transactions]
total_item_occurrences = sum(transaction_lengths)
mean_transaction_length = total_item_occurrences / n_transactions
sorted_transaction_lengths = sort(transaction_lengths)
median_transaction_length = (sorted_transaction_lengths[n_transactions ÷ 2] +
    sorted_transaction_lengths[n_transactions ÷ 2 + 1]) / 2
transaction_length_sd = sqrt(sum((value - mean_transaction_length)^2 for value in transaction_lengths) / n_transactions)
transaction_density = total_item_occurrences / (n_transactions * 32)
singleton_support = Dict(item => 0 for item in 1:32)
for transaction in transactions, item in transaction
    singleton_support[item] += 1
end

# Reload class metadata in this Julia process; notebook cells do not share Julia memory.
metadata_path = artifact_path("image_metadata")
metadata_sha = bytes2hex(open(sha256, metadata_path))
metadata_sha == lowercase(String(transaction_artifacts["image_metadata"]["sha256"])) ||
    error("Image metadata changed after bundle validation.")
class_for_image = Dict{String,Int}()
open(metadata_path, "r") do io
    readline(io) == "image_id,split,source_index,class_label" ||
        error("Unexpected verified image metadata header.")
    for line in eachline(io)
        fields = split(line, ',')
        length(fields) == 4 || error("Malformed image metadata row.")
        image_id = fields[1]
        split_name, source_index = parse_image_id(image_id)
        expected_split, expected_index = split_name == :train ?
            ("train", source_index) : ("test", source_index)
        fields[2] == expected_split && parse(Int, fields[3]) == expected_index ||
            error("Image metadata identity differs from its image ID.")
        label = parse(Int, fields[4])
        0 <= label <= 9 || error("Image class label is outside 0–9.")
        class_for_image[image_id] = label
    end
end
length(class_for_image) == 70_000 || error("Expected metadata for 70,000 images.")

### Vẽ phân bố giao dịch và tần suất từ

Tạo hai biểu đồ SVG từ histogram độ dài giao dịch và số giao dịch chứa từng từ thị giác, rồi hiển thị chúng.

In [ ]:
function write_length_svg(path, histogram)
    maximum_length = maximum(keys(histogram))
    width, height, left, top, bottom = 980, 370, 58, 22, 48
    plot_width = width - left - 24
    plot_height = height - top - bottom
    max_value = maximum(values(histogram))
    bar_step = plot_width / (maximum_length + 1)
    bar_width = max(2.0, bar_step - 3.0)
    open(path, "w") do io
        println(io, "<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"$width\" height=\"$height\" viewBox=\"0 0 $width $height\">")
        println(io, "<rect width=\"100%\" height=\"100%\" fill=\"white\"/>")
        println(io, "<text x=\"$(width/2)\" y=\"18\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"15\">Độ dài giao dịch, kể cả giao dịch rỗng</text>")
        println(io, "<line x1=\"$left\" y1=\"$(top+plot_height)\" x2=\"$(left+plot_width)\" y2=\"$(top+plot_height)\" stroke=\"#333\"/>")
        for length_value in 0:maximum_length
            value = get(histogram, length_value, 0)
            x = left + length_value*bar_step + 1
            bar_height = max_value == 0 ? 0.0 : value/max_value*plot_height
            y = top + plot_height - bar_height
            println(io, "<rect x=\"$x\" y=\"$y\" width=\"$bar_width\" height=\"$bar_height\" fill=\"#315c9b\"><title>Độ dài $length_value: $value</title></rect>")
            length_value % 2 == 0 && println(io, "<text x=\"$(x+bar_width/2)\" y=\"$(top+plot_height+18)\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"10\">$length_value</text>")
        end
        println(io, "<text x=\"$(width/2)\" y=\"$(height-7)\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"12\">Số từ thị giác trong một giao dịch</text></svg>")
    end
end

function write_singleton_svg(path, supports, n)
    width, height, left, top, bottom = 980, 370, 58, 22, 52
    plot_width = width-left-24
    plot_height = height-top-bottom
    max_value = maximum(values(supports))
    step = plot_width / length(supports)
    bar_width = step - 4
    open(path, "w") do io
        println(io, "<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"$width\" height=\"$height\" viewBox=\"0 0 $width $height\">")
        println(io, "<rect width=\"100%\" height=\"100%\" fill=\"white\"/>")
        println(io, "<text x=\"$(width/2)\" y=\"18\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"15\">Tần suất từng từ thị giác</text>")
        println(io, "<line x1=\"$left\" y1=\"$(top+plot_height)\" x2=\"$(left+plot_width)\" y2=\"$(top+plot_height)\" stroke=\"#333\"/>")
        for item in 1:32
            value = supports[item]
            x = left + (item-1)*step + 2
            bar_height = max_value == 0 ? 0.0 : value/max_value*plot_height
            y = top + plot_height - bar_height
            println(io, "<rect x=\"$x\" y=\"$y\" width=\"$bar_width\" height=\"$bar_height\" fill=\"#26856a\"><title>Từ thị giác $item: $value giao dịch ($(round(100value/n; digits=2))%)</title></rect>")
            println(io, "<text x=\"$(x+bar_width/2)\" y=\"$(top+plot_height+16)\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"9\">$item</text>")
        end
        println(io, "<text x=\"$(width/2)\" y=\"$(height-7)\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"12\">Mã từ thị giác</text></svg>")
    end
end
write_length_svg(joinpath(output_dir, "transaction_lengths.svg"), length_histogram)
write_singleton_svg(joinpath(output_dir, "visual_word_prevalence.svg"), singleton_support, n_transactions)

function markdown_table(rows, headers)
    io = IOBuffer()
    println(io, "| ", join(headers, " | "), " |")
    println(io, "|", join(fill("---", length(headers)), "|"), "|")
    for row in rows
        println(io, "| ", join(string.(row), " | "), " |")
    end
    return String(take!(io))
end
function csv_value(value)
    raw = ismissing(value) ? "" : string(value)
    needs_quotes = occursin(',', raw) || occursin('"', raw) ||
        occursin('\n', raw) || occursin('\r', raw)
    return needs_quotes ? "\"" * replace(raw, "\"" => "\"\"") * "\"" : raw
end

function write_csv(path, headers, rows)
    open(path, "w") do io
        println(io, join(csv_value.(headers), ','))
        for row in rows
            fields = row isa NamedTuple ? values(row) : row
            println(io, join(csv_value.(fields), ','))
        end
    end
    return path
end

length_rows = [(key, get(length_histogram,key,0)) for key in 0:maximum(keys(length_histogram))]

write_csv(joinpath(output_dir, "transaction_length_distribution.csv"),
    ("transaction_length", "transaction_count"), length_rows)
write_csv(joinpath(output_dir, "visual_word_frequency.csv"),
    ("visual_word_id", "transaction_count", "transaction_percent"),
    [(item, singleton_support[item], round(100singleton_support[item]/n_transactions; digits=4)) for item in 1:32])
write_csv(joinpath(output_dir, "transaction_statistics.csv"),
    ("transaction_count", "empty_transactions", "total_item_occurrences", "mean_length",
     "median_length", "population_sd", "density_percent"),
    [(n_transactions, input.empty_count, total_item_occurrences,
      round(mean_transaction_length; digits=8), median_transaction_length,
      round(transaction_length_sd; digits=8), round(100transaction_density; digits=6))])


display(Markdown.parse("Phân bố độ dài tính cả giao dịch rỗng."))
show_demo_image(joinpath(output_dir, "transaction_lengths.svg"); mime="image/svg+xml", alt="Phân bố độ dài giao dịch")
display(Markdown.parse("Tần suất là số giao dịch chứa mỗi từ thị giác trong 70.000 ảnh."))
show_demo_image(joinpath(output_dir, "visual_word_prevalence.svg"); mime="image/svg+xml", alt="Tần suất từ thị giác")


## 5. Đối chiếu Eclat Set và BitVector

Với mỗi ngưỡng hỗ trợ, notebook chạy cả hai backend trên cùng giao dịch rồi so sánh toàn bộ tập mục và số hỗ trợ. Lỗi chạy được ghi là `ERROR`; không được diễn giải thành kết quả rỗng.

In [ ]:
thresholds = (80, 60, 40, 20, 10, 5)
mining_rows = NamedTuple[]
mining_results_by_percent = Dict{Int,Any}()
mining_backend_results_by_percent = Dict{Int,Any}()
five_percent_result = nothing
for percent in thresholds
    min_count = cld(percent * n_transactions, 100)
    set_result = nothing
    bit_result = nothing
    set_error = ""
    bit_error = ""
    try
        set_result = EclatFIM.mine(transactions, min_count; backend=:set)
    catch err
        err isa InterruptException && rethrow()
        set_error = sprint(showerror, err)
    end
    try
        bit_result = EclatFIM.mine(transactions, min_count; backend=:bitvector)
    catch err
        err isa InterruptException && rethrow()
        bit_error = sprint(showerror, err)
    end
    set_status = isempty(set_error) ? "COMPLETED" : "ERROR"
    bit_status = isempty(bit_error) ? "COMPLETED" : "ERROR"
    parity = set_status == "COMPLETED" && bit_status == "COMPLETED" ?
        (set_result == bit_result ? "PASS" : "MISMATCH") : "NOT COMPARED"
    set_count = set_status == "COMPLETED" ? length(set_result) : missing
    bit_count = bit_status == "COMPLETED" ? length(bit_result) : missing
    push!(mining_rows, (; percent, min_count, set_status, set_count, bit_status, bit_count,
        parity, set_error, bit_error))
    if parity == "PASS"
        mining_results_by_percent[percent] = set_result
        mining_backend_results_by_percent[percent] = (; set=set_result, bitvector=bit_result)
    end
    percent == 5 && parity == "PASS" && (five_percent_result = set_result)
    println("Support $(percent)% (min_count=$(min_count)): Set=$(set_status), BitVector=$(bit_status), parity=$(parity)")
    isempty(set_error) || println("  Set error: ", set_error)
    isempty(bit_error) || println("  BitVector error: ", bit_error)
end

## 6. Đo thời gian và peak RSS trong tiến trình Julia cô lập

Mỗi backend ở mỗi ngưỡng được chạy một lần trong tiến trình Julia mới bằng worker có sẵn của dự án. Worker đọc JSONL, khai thác rồi ghi kết quả; notebook đối chiếu toàn bộ tập mục với kết quả trong kernel trước khi nhận số đo. Thời gian gồm đọc giao dịch, khai thác và ghi kết quả. Peak RSS là mức cao nhất của toàn tiến trình worker, gồm cả Julia và bước đọc dữ liệu; đây không phải lượng bộ nhớ riêng của thuật toán. Mỗi lượt là phép đo đơn, không warm-up hay lặp, nên số liệu phụ thuộc máy chạy.



In [ ]:
using TOML

worker_script = joinpath(repo_root, "src", "applications", "Fashion-Mnist", "mining", "scripts", "run_mining_worker.jl")
isfile(worker_script) || error("Julia Eclat worker is missing: $worker_script")
benchmark_dir = joinpath(output_dir, "eclat_measurements")
mkpath(benchmark_dir)
worker_output_limit = 512_000_000
eclat_benchmark_rows = NamedTuple[]

function read_worker_itemsets(path)
    result = Dict{Tuple{Vararg{Int}},Int}()
    for line in eachline(path)
        items_marker = "\"items\":["
        start_range = findfirst(items_marker, line)
        start_range === nothing && error("Malformed worker itemset row.")
        items_start = last(start_range) + 1
        items_end = findnext(']', line, items_start)
        items_end === nothing && error("Malformed worker item list.")
        item_text = line[items_start:prevind(line, items_end)]
        items = isempty(item_text) ? Int[] : parse.(Int, split(item_text, ','))
        length_marker = "\"length\":"
        support_marker = "\"support_count\":"
        length_range = findfirst(length_marker, line)
        support_range = findfirst(support_marker, line)
        length_range === nothing && error("Worker row has no itemset length.")
        support_range === nothing && error("Worker row has no support count.")
        reported_length = parse(Int, split(line[nextind(line, last(length_range)):end], ',')[1])
        support_count = parse(Int, split(line[nextind(line, last(support_range)):end], ',')[1])
        length(items) == reported_length || error("Worker itemset length is inconsistent.")
        result[Tuple(items)] = support_count
    end
    return result
end

for percent in thresholds
    min_count = cld(percent * n_transactions, 100)
    expected = get(mining_backend_results_by_percent, percent, nothing)
    for backend in (:set, :bitvector)
        status = "NOT COMPARED"
        runtime_seconds = missing
        peak_rss_mb = missing
        pattern_check = "NOT CHECKED"
        error_message = ""
        if isnothing(expected)
            push!(eclat_benchmark_rows, (; percent, backend, min_count, status,
                runtime_seconds, peak_rss_mb, pattern_check, error_message))
            continue
        end

        run_dir = joinpath(benchmark_dir, "$(percent)_$(backend)")
        mkpath(run_dir)
        staging_path = joinpath(run_dir, "itemsets.jsonl")
        result_path = joinpath(run_dir, "worker_result.toml")
        log_path = joinpath(run_dir, "worker.log")
        args = [worker_script, transactions_path, staging_path, result_path,
            string(min_count), string(worker_output_limit), string(n_transactions),
            "--backend", string(backend)]
        command = Cmd(vcat(Base.julia_cmd().exec, ["--project=$(repo_root)"], args))
        command = addenv(command, "JULIA_NUM_THREADS"=>"1",
            "OPENBLAS_NUM_THREADS"=>"1", "OPENBLAS_CORETYPE"=>"Haswell",
            "JULIA_PKG_PRECOMPILE_AUTO"=>"0")
        try
            process = open(log_path, "w") do log_io
                run(pipeline(ignorestatus(command), stdout=log_io, stderr=log_io))
            end
            isfile(result_path) || error("Worker exited with code $(process.exitcode) and wrote no metrics.")
            metrics = TOML.parsefile(result_path)
            worker_status = String(get(metrics, "worker_status", "MISSING_STATUS"))
            if worker_status in ("COMPLETED_FULL", "COMPLETED_ZERO") && isfile(staging_path)
                worker_patterns = read_worker_itemsets(staging_path)
                expected_patterns = backend == :set ? expected.set : expected.bitvector
                pattern_check = worker_patterns == expected_patterns ? "PASS" : "MISMATCH"
                valid_record_count = Int(metrics["record_count"]) == length(expected_patterns)
                valid_digest = lowercase(String(metrics["input_sha256"])) == lowercase(String(input.input_sha256))
                if pattern_check == "PASS" && valid_record_count && valid_digest
                    status = "PASS"
                    runtime_seconds = Float64(metrics["worker_runtime_seconds"])
                    rss = get(metrics, "peak_rss_mb", "NA")
                    peak_rss_mb = rss isa Number ? Float64(rss) : missing
                else
                    status = "VERIFY FAILED"
                    error_message = "Worker output, record count, or input checksum differs from the kernel result."
                end
            else
                status = worker_status
                error_message = isfile(log_path) ? join(readlines(log_path)[max(1,end-4):end], " ") : "Worker did not complete."
            end
            process.exitcode == 0 || (status = "WORKER EXIT $(process.exitcode)")
        catch err
            status = "ERROR"
            error_message = sprint(showerror, err)
        end
        push!(eclat_benchmark_rows, (; percent, backend, min_count, status,
            runtime_seconds, peak_rss_mb, pattern_check, error_message))
        println("$(percent)% $(backend): $(status), itemsets=$(pattern_check)")
    end
end


### Tổng hợp kết quả khai thác và phép đo

Lập bảng đối chiếu Set/BitVector, thêm thời gian và peak RSS từ worker Julia cô lập, rồi ghi tóm tắt giao dịch và kết quả khai thác.


In [ ]:
mining_table_rows = [
    (string(row.percent, "%"), row.min_count, row.set_status,
        ismissing(row.set_count) ? "—" : row.set_count,
        row.bit_status, ismissing(row.bit_count) ? "—" : row.bit_count, row.parity)
    for row in mining_rows
]
fit_converged = Bool(fit_metrics["converged"])
fit_iterations = Int(fit_metrics["iterations"])
fit_limit = Int(fit_metrics["max_iterations"])
fit_occupancy = get(fit_metrics, "occupancy_used", missing)
fit_entropy = get(fit_metrics, "normalized_entropy", missing)
fit_distortion = get(fit_metrics, "distortion_mean_squared", missing)
fit_top_share = get(fit_metrics, "top_cluster_share", missing)
fit_seconds = get(fit_metrics, "fit_seconds", missing)
fit_stop_reason = get(fit_metrics, "stop_reason", "not_recorded")
codebook_sha = lowercase(String(transaction_artifacts["codebook"]["sha256"]))
pass_count = count(row -> row.parity == "PASS", mining_rows)
mismatch_count = count(row -> row.parity == "MISMATCH", mining_rows)
uncompared_count = count(row -> row.parity != "PASS" && row.parity != "MISMATCH", mining_rows)
five_percent_row = only(filter(row -> row.percent == 5, mining_rows))
pattern_length_counts = Dict{Int,Int}()
if !isnothing(five_percent_result)
    for itemset in keys(five_percent_result)
        length_value = length(itemset)
        pattern_length_counts[length_value] = get(pattern_length_counts, length_value, 0) + 1
    end
end
pattern_length_rows = [(length_value, pattern_length_counts[length_value])
    for length_value in sort(collect(keys(pattern_length_counts)))]
max_pattern_length = isempty(pattern_length_counts) ? 0 : maximum(keys(pattern_length_counts))

write_csv(joinpath(output_dir, "itemset_length_distribution.csv"),
    ("itemset_length", "itemset_count"), pattern_length_rows)
benchmark_lookup = Dict((row.percent, row.backend) => row for row in eclat_benchmark_rows)
eclat_csv_rows = []
for row in mining_rows, backend in (:set, :bitvector)
    benchmark = get(benchmark_lookup, (row.percent, backend), nothing)
    itemset_count = backend == :set ? row.set_count : row.bit_count
    backend_status = backend == :set ? row.set_status : row.bit_status
    push!(eclat_csv_rows, (row.percent, row.min_count, string(backend), backend_status,
        itemset_count, row.parity,
        isnothing(benchmark) ? missing : benchmark.status,
        isnothing(benchmark) ? missing : benchmark.runtime_seconds,
        isnothing(benchmark) ? missing : benchmark.peak_rss_mb,
        isnothing(benchmark) ? missing : benchmark.pattern_check))
end
write_csv(joinpath(output_dir, "eclat_results_and_benchmarks.csv"),
    ("minsup_percent", "min_support_count", "backend", "mining_status", "itemset_count",
     "in_kernel_parity", "worker_status", "worker_runtime_seconds", "worker_peak_rss_mb",
     "worker_output_check"), eclat_csv_rows)

verified_thresholds = sort(collect(keys(mining_results_by_percent)); rev=true)
threshold_nested = length(verified_thresholds) < 2 || all(
    all(itemset -> haskey(mining_results_by_percent[verified_thresholds[i+1]], itemset),
        keys(mining_results_by_percent[verified_thresholds[i]]))
    for i in 1:length(verified_thresholds)-1)

if fit_converged
    fit_interpretation = "K-means báo hội tụ sau $(fit_iterations)/$(fit_limit) vòng. Đây là trạng thái của lượt trong gói, không phải bằng chứng rằng codebook tối ưu theo mọi tiêu chí."
else
    fit_interpretation = "K-means chưa hội tụ trong giới hạn $(fit_limit) vòng (đã chạy $(fit_iterations) vòng). Cần đọc trạng thái kiểm định giao dịch riêng; không diễn giải việc chạm trần là hội tụ."
end

if five_percent_row.parity == "PASS" && five_percent_row.set_count == 0
    five_percent_interpretation = "Ở ngưỡng 5%, cả hai backend hoàn tất và cùng trả về 0 tập mục; không có mẫu để truy ảnh."
elseif five_percent_row.parity == "PASS"
    five_percent_interpretation = "Ở ngưỡng 5%, hai backend khớp trên toàn bộ tập mục và số hỗ trợ; notebook chọn mẫu theo quy tắc cố định ở phần dưới."
elseif five_percent_row.parity == "MISMATCH"
    five_percent_interpretation = "Ở ngưỡng 5%, hai backend không khớp; không dùng mẫu của lượt này để diễn giải ảnh."
else
    five_percent_interpretation = "Ở ngưỡng 5%, chưa có cặp kết quả hoàn tất để đối chiếu; không xem trạng thái này là kết quả rỗng."
end

open(joinpath(output_dir, "mining_summary.md"), "w") do io
    println(io, "## Lượt đang phân tích")
    println(io)
    println(io, "- Mã lượt: ", demo["run_id"])
    println(io, "- Mã lượt từ điển: ", demo["tc_run_id"])
    println(io, "- SHA-256 giao dịch: ", input.input_sha256)
    println(io, "- SHA-256 codebook: ", codebook_sha)
    println(io, "- Giao dịch: ", n_transactions, "; giao dịch rỗng: ", input.empty_count, ".")
    println(io)
    println(io, "## Diễn giải kết quả của lượt này")
    println(io)
    println(io, "- K-means: ", fit_metrics["status"], "; hội tụ: ", fit_converged,
        "; số vòng: ", fit_iterations, "/", fit_limit, "; K=", fit_metrics["k"], "; seed=", fit_metrics["seed"], ".")
    println(io, "- ", fit_interpretation)
    if !ismissing(fit_occupancy)
        println(io, "- Codebook có phần tử ở ", fit_occupancy, "/", fit_metrics["k"], " cụm.")
    end
    if !ismissing(fit_entropy)
        println(io, "- Entropy chuẩn hóa của codebook: ", round(Float64(fit_entropy); digits=4), ".")
    end
    if !ismissing(fit_distortion)
        println(io, "- Độ méo bình phương trung bình theo manifest: ", round(Float64(fit_distortion); digits=6), ".")
    end
    if !ismissing(fit_top_share)
        println(io, "- Tỷ trọng descriptor thuộc cụm lớn nhất: ", round(100Float64(fit_top_share); digits=2), "%; lý do dừng: ", fit_stop_reason, ".")
    end
    if !ismissing(fit_seconds)
        println(io, "- Thời gian fit được ghi trong manifest: ", round(Float64(fit_seconds); digits=2), " giây.")
    end
    println(io, "- Số lượt xuất hiện của từ thị giác: ", total_item_occurrences,
        "; độ dài trung bình: ", round(mean_transaction_length; digits=4),
        "; trung vị: ", median_transaction_length,
        "; độ lệch chuẩn tổng thể: ", round(transaction_length_sd; digits=4),
        "; mật độ trên 32 mã: ", round(100transaction_density; digits=2), "%."
    )
    println(io, "- Eclat: ", pass_count, "/", length(mining_rows), " ngưỡng đối chiếu PASS; ",
        mismatch_count, " MISMATCH; ", uncompared_count, " ngưỡng chưa đối chiếu.")
    println(io, "- ", five_percent_interpretation)
    println(io, "- Tập mục ở các ngưỡng đã đối chiếu có quan hệ bao hàm theo chiều hạ minsup: ", threshold_nested, ".")
    println(io, "- Ở ngưỡng 5%, tập dài nhất có ", max_pattern_length, " mã; phân bố độ dài được ghi ở bảng bên dưới.")
    println(io, "- Các phân tích bên dưới dùng đúng gói có checksum codebook và giao dịch ở trên. Nếu codebook khác, hãy lấy gói mới từ notebook 02; notebook này sẽ tính lại các bảng và hình.")
    println(io)
    println(io, "## Phân bố độ dài giao dịch")
    println(io)
    println(io, markdown_table(length_rows, ("Số từ thị giác", "Số giao dịch")))
    println(io)
    println(io, "## Đối chiếu Eclat")
    println(io)
    println(io, markdown_table(mining_table_rows,
        ("Ngưỡng hỗ trợ", "Số hỗ trợ tối thiểu", "Set", "Số tập mục", "BitVector", "Số tập mục", "Đối chiếu")))
    println(io)
    println(io, "Đối chiếu PASS nghĩa là toàn bộ tập mục và số hỗ trợ khớp. Một lượt lỗi được ghi ERROR và không được xem là kết quả rỗng.")
    println(io)
    println(io, "## Độ dài tập mục ở ngưỡng 5%")
    println(io)
    println(io, markdown_table(pattern_length_rows, ("Số mã trong tập mục", "Số tập mục")))
    println(io)
    println(io, "## Thời gian và peak RSS của Eclat")
    println(io)
    benchmark_table_rows = [
        (string(row.percent, "%"), string(row.backend), row.min_count, row.status,
            ismissing(row.runtime_seconds) ? "—" : round(Float64(row.runtime_seconds); digits=3),
            ismissing(row.peak_rss_mb) ? "—" : round(Float64(row.peak_rss_mb); digits=2),
            row.pattern_check)
        for row in eclat_benchmark_rows
    ]
    println(io, markdown_table(benchmark_table_rows,
        ("Ngưỡng", "Backend", "Số hỗ trợ tối thiểu", "Trạng thái", "Thời gian (giây)", "Peak RSS (MiB)", "Đối chiếu đầu ra")))
    println(io)
    println(io, "Worker dùng tiến trình Julia riêng cho từng backend và ngưỡng. Thời gian tính từ lúc đọc JSONL đến lúc ghi tập mục; peak RSS là mức cao nhất của toàn worker, gồm Julia và dữ liệu đầu vào. Đây là phép đo đơn, không warm-up hay lặp; không xem RSS là bộ nhớ riêng của Eclat.")
    println(io)
    println(io, "## Tần suất từ thị giác")
    println(io)
    println(io, markdown_table([(item, singleton_support[item], round(100singleton_support[item]/n_transactions; digits=2)) for item in 1:32],
        ("Từ thị giác", "Số giao dịch", "Phần trăm giao dịch")))
end
show_demo_markdown(joinpath(output_dir, "mining_summary.md"))


## 7. Phân tích làm giàu trên toàn bộ tập mục

Notebook tính số ảnh hỗ trợ của từng tập mục ở mỗi lớp, đối chiếu tổng theo lớp với hỗ trợ Eclat, rồi tổng hợp hệ số làm giàu ở cả sáu ngưỡng. Phần nhãn chỉ được dùng sau khi mining. Các thống kê mô tả gói 70.000 ảnh, không phải đánh giá phân loại độc lập.


In [ ]:
if !isnothing(five_percent_result) && !isempty(five_percent_result)
    pattern_pairs = sort(collect(five_percent_result); by=pair -> (length(first(pair)), first(pair)))
    all_patterns = [first(pair) for pair in pattern_pairs]
    all_pattern_supports = [Int(last(pair)) for pair in pattern_pairs]
    n_patterns = length(all_patterns)

    class_positions = zeros(Int, n_transactions)
    class_population = zeros(Int, 10)
    for tid in 1:n_transactions
        label = class_for_image[input.image_id[tid]]
        class_population[label+1] += 1
        class_positions[tid] = class_population[label+1]
    end
    sum(class_population) == n_transactions || error("Class populations do not cover all transactions.")
    all(==(7_000), class_population) || error("Expected 7,000 images in each class.")

    class_item_masks = [
        Dict(item => zeros(UInt64, cld(class_population[class_id], 64)) for item in 1:32)
        for class_id in 1:10
    ]
    for tid in 1:n_transactions
        class_id = class_for_image[input.image_id[tid]] + 1
        local_position = class_positions[tid]
        word_index = ((local_position - 1) >>> 6) + 1
        bit_value = UInt64(1) << ((local_position - 1) & 63)
        for item in transactions[tid]
            class_item_masks[class_id][item][word_index] |= bit_value
        end
    end

    class_support_matrix = zeros(Int, n_patterns, 10)
    for (pattern_index, itemset) in enumerate(all_patterns)
        for class_id in 1:10
            masks = class_item_masks[class_id]
            scratch = copy(masks[first(itemset)])
            for item in Iterators.drop(itemset, 1)
                other = masks[item]
                @inbounds for word_index in eachindex(scratch)
                    scratch[word_index] &= other[word_index]
                end
            end
            class_support_matrix[pattern_index, class_id] = sum(count_ones, scratch)
        end
        sum(@view class_support_matrix[pattern_index, :]) == all_pattern_supports[pattern_index] ||
            error("Class supports do not sum to Eclat support for $(all_patterns[pattern_index]).")
    end

    class_rate_matrix = class_support_matrix ./ reshape(Float64.(class_population), 1, :)
    overall_rate_vector = all_pattern_supports ./ n_transactions
    class_enrichment_matrix = class_rate_matrix ./ reshape(overall_rate_vector, :, 1)
    max_enrichment = [maximum(@view class_enrichment_matrix[index, :]) for index in 1:n_patterns]
    pattern_index_by_tuple = Dict(itemset => index for (index, itemset) in enumerate(all_patterns))

    function notebook_median(values)
        ordered = sort(Float64.(collect(values)))
        isempty(ordered) && return missing
        middle = length(ordered) ÷ 2
        return isodd(length(ordered)) ? ordered[middle+1] : (ordered[middle] + ordered[middle+1]) / 2
    end
    function notebook_quantile(ordered, probability)
        position = 1 + (length(ordered) - 1) * probability
        low, high = floor(Int, position), ceil(Int, position)
        fraction = position - low
        return ordered[low] * (1 - fraction) + ordered[high] * fraction
    end

    enrichment_threshold_rows = NamedTuple[]
    for percent in thresholds
        min_count = cld(percent * n_transactions, 100)
        indexes = findall(index -> all_pattern_supports[index] >= min_count, 1:n_patterns)
        values = max_enrichment[indexes]
        enriched_count = count(value -> value >= 2.0, values)
        push!(enrichment_threshold_rows, (;
            percent, min_count, itemset_count=length(indexes),
            median_emax=notebook_median(values), enriched_count,
            enriched_fraction=isempty(values) ? missing : enriched_count / length(values)))
    end
    class_enriched_counts = [
        count(index -> class_enrichment_matrix[index, class_id] >= 2.0, 1:n_patterns)
        for class_id in 1:10
    ]

    function jaccard_similarity(left, right)
        common = count(item -> item in right, left)
        return common / (length(left) + length(right) - common)
    end
    selected_pattern_indexes = Int[]
    function add_diverse_pattern!(candidates)
        available = [index for index in candidates if !(index in selected_pattern_indexes)]
        isempty(available) && return false
        diverse = [index for index in available if all(
            jaccard_similarity(all_patterns[index], all_patterns[chosen]) <= 0.80
            for chosen in selected_pattern_indexes)]
        pool = isempty(diverse) ? available : diverse
        sort!(pool; by=index -> (-all_pattern_supports[index], all_patterns[index]))
        push!(selected_pattern_indexes, first(pool))
        return true
    end
    add_diverse_pattern!(findall(index -> length(all_patterns[index]) == 1, 1:n_patterns))
    add_diverse_pattern!(findall(index -> length(all_patterns[index]) == 2, 1:n_patterns))
    add_diverse_pattern!(findall(index -> length(all_patterns[index]) >= 3, 1:n_patterns))
    add_diverse_pattern!(findall(index ->
        length(all_patterns[index]) >= 2 &&
        0.05 <= all_pattern_supports[index] / n_transactions < 0.10, 1:n_patterns))
    while length(selected_pattern_indexes) < min(4, n_patterns)
        add_diverse_pattern!(collect(1:n_patterns)) || break
    end
    example_patterns = [all_patterns[index] for index in selected_pattern_indexes]

    example_rows = []
    for index in selected_pattern_indexes
        class_id = argmax(@view class_enrichment_matrix[index, :])
        push!(example_rows, ("{" * join(all_patterns[index], ",") * "}",
            all_pattern_supports[index],
            round(100overall_rate_vector[index]; digits=3),
            FASHION_CLASSES[class_id],
            round(100class_rate_matrix[index, class_id]; digits=2),
            round(class_enrichment_matrix[index, class_id]; digits=3)))
    end
    class_count_rows = [
        (FASHION_CLASSES[class_id], class_population[class_id], class_enriched_counts[class_id])
        for class_id in 1:10
    ]
    threshold_rows = [
        (string(row.percent, "%"), row.min_count, row.itemset_count,
            ismissing(row.median_emax) ? "—" : round(Float64(row.median_emax); digits=3),
            row.enriched_count,
            ismissing(row.enriched_fraction) ? "—" : string(round(100Float64(row.enriched_fraction); digits=2), "%"))
        for row in enrichment_threshold_rows
    ]

    heatmap_rows = NamedTuple[]
    for target_class in 1:10
        candidates = findall(index ->
            length(all_patterns[index]) >= 2 &&
            class_support_matrix[index, target_class] >= cld(class_population[target_class], 10),
            1:n_patterns)
        isempty(candidates) && continue
        sort!(candidates; by=index ->
            (-class_enrichment_matrix[index, target_class],
             -all_pattern_supports[index], all_patterns[index]))
        push!(heatmap_rows, (; target_class, pattern_index=first(candidates)))
    end

    function svg_hex(value)
        return lpad(lowercase(string(clamp(round(Int, value), 0, 255), base=16)), 2, '0')
    end
    function svg_color(rate_percent)
        amount = clamp(rate_percent / 100, 0.0, 1.0)
        red = 248 * (1 - amount) + 68 * amount
        green = 248 * (1 - amount) + 46 * amount
        blue = 252 * (1 - amount) + 132 * amount
        return "#" * svg_hex(red) * svg_hex(green) * svg_hex(blue)
    end
    function svg_escape(value)
        return replace(string(value), "&" => "&amp;", "<" => "&lt;", ">" => "&gt;")
    end

    boxplot_path = joinpath(output_dir, "enrichment_by_threshold.svg")
    drawable_rows = [row for row in enrichment_threshold_rows if row.itemset_count > 0]
    y_max = max(2.5, maximum(max_enrichment))
    width, height, left, top, bottom = 920, 430, 76, 42, 74
    plot_width, plot_height = width-left-30, height-top-bottom
    ycoord(value) = top + plot_height - clamp(value/y_max, 0.0, 1.0)*plot_height
    open(boxplot_path, "w") do io
        println(io, "<svg xmlns='http://www.w3.org/2000/svg' width='$width' height='$height' viewBox='0 0 $width $height'>")
        println(io, "<rect width='100%' height='100%' fill='white'/>")
        println(io, "<text x='$(width/2)' y='23' text-anchor='middle' font-family='sans-serif' font-size='18'>Phân bố hệ số làm giàu lớn nhất theo ngưỡng</text>")
        for tick in range(0, y_max; length=6)
            y = ycoord(tick)
            println(io, "<line x1='$left' y1='$y' x2='$(left+plot_width)' y2='$y' stroke='#e1e1e1'/>")
            println(io, "<text x='$(left-10)' y='$(y+4)' text-anchor='end' font-family='sans-serif' font-size='12'>$(round(tick; digits=1))</text>")
        end
        y = ycoord(2.0)
        println(io, "<line x1='$left' y1='$y' x2='$(left+plot_width)' y2='$y' stroke='#bd4b45' stroke-dasharray='6,5'/>")
        println(io, "<text x='$(left+plot_width-2)' y='$(y-6)' text-anchor='end' font-family='sans-serif' font-size='12' fill='#a3332e'>E = 2</text>")
        step = plot_width / max(length(drawable_rows), 1)
        for (position, row) in enumerate(drawable_rows)
            indexes = findall(index -> all_pattern_supports[index] >= row.min_count, 1:n_patterns)
            values = sort(max_enrichment[indexes])
            x = left + (position - 0.5) * step
            low_y, q1_y = ycoord(first(values)), ycoord(notebook_quantile(values, 0.25))
            median_y, q3_y, high_y = ycoord(notebook_median(values)), ycoord(notebook_quantile(values, 0.75)), ycoord(last(values))
            println(io, "<line x1='$x' y1='$low_y' x2='$x' y2='$high_y' stroke='#243b64' stroke-width='2'/>")
            println(io, "<line x1='$(x-12)' y1='$low_y' x2='$(x+12)' y2='$low_y' stroke='#243b64' stroke-width='2'/>")
            println(io, "<line x1='$(x-12)' y1='$high_y' x2='$(x+12)' y2='$high_y' stroke='#243b64' stroke-width='2'/>")
            box_top, box_bottom = min(q1_y, q3_y), max(q1_y, q3_y)
            println(io, "<rect x='$(x-22)' y='$box_top' width='44' height='$(max(1, box_bottom-box_top))' fill='#bfd5ef' stroke='#243b64'/>")
            println(io, "<line x1='$(x-22)' y1='$median_y' x2='$(x+22)' y2='$median_y' stroke='#b03a2e' stroke-width='3'/>")
            println(io, "<text x='$x' y='$(top+plot_height+24)' text-anchor='middle' font-family='sans-serif' font-size='13'>$(row.percent)%</text>")
        end
        println(io, "<text x='$(left+plot_width/2)' y='$(height-12)' text-anchor='middle' font-family='sans-serif' font-size='13'>Ngưỡng minsup; râu biểu đồ min–max</text>")
        println(io, "<text x='18' y='$(top+plot_height/2)' transform='rotate(-90 18 $(top+plot_height/2))' text-anchor='middle' font-family='sans-serif' font-size='13'>Emax</text></svg>")
    end

    class_count_path = joinpath(output_dir, "patterns_enriched_by_class.svg")
    width, height, left, top, bottom = 980, 400, 76, 40, 74
    plot_width, plot_height = width-left-28, height-top-bottom
    largest_class_count = max(1, maximum(class_enriched_counts))
    open(class_count_path, "w") do io
        println(io, "<svg xmlns='http://www.w3.org/2000/svg' width='$width' height='$height' viewBox='0 0 $width $height'>")
        println(io, "<rect width='100%' height='100%' fill='white'/>")
        println(io, "<text x='$(width/2)' y='22' text-anchor='middle' font-family='sans-serif' font-size='17'>Số tập mục có E(X,c) ≥ 2 ở ngưỡng 5%</text>")
        println(io, "<line x1='$left' y1='$(top+plot_height)' x2='$(left+plot_width)' y2='$(top+plot_height)' stroke='#333'/>")
        step = plot_width / 10
        for class_id in 1:10
            value = class_enriched_counts[class_id]
            bar_height = value/largest_class_count*plot_height
            x = left+(class_id-1)*step+4
            y = top+plot_height-bar_height
            bar_width = step-8
            println(io, "<rect x='$x' y='$y' width='$bar_width' height='$bar_height' fill='#7652a6'><title>$(svg_escape(FASHION_CLASSES[class_id])): $value tập</title></rect>")
            println(io, "<text x='$(x+bar_width/2)' y='$(top+plot_height+17)' text-anchor='middle' font-family='sans-serif' font-size='9'>$(class_id-1)</text>")
            println(io, "<text x='$(x+bar_width/2)' y='$(y-5)' text-anchor='middle' font-family='sans-serif' font-size='9'>$value</text>")
        end
        println(io, "<text x='$(left+plot_width/2)' y='$(height-10)' text-anchor='middle' font-family='sans-serif' font-size='12'>Nhãn lớp theo thứ tự 0–9</text></svg>")
    end

    heatmap_path = joinpath(output_dir, "class_pattern_heatmap.svg")
    cell_width, cell_height, left, top = 72, 34, 300, 84
    width, height = left+10*cell_width+20, top+max(length(heatmap_rows),1)*cell_height+22
    open(heatmap_path, "w") do io
        println(io, "<svg xmlns='http://www.w3.org/2000/svg' width='$width' height='$height' viewBox='0 0 $width $height'>")
        println(io, "<rect width='100%' height='100%' fill='white'/>")
        println(io, "<text x='$(width/2)' y='23' text-anchor='middle' font-family='sans-serif' font-size='17'>Tỷ lệ ảnh theo lớp chứa mẫu được chọn</text>")
        println(io, "<text x='$(width/2)' y='47' text-anchor='middle' font-family='sans-serif' font-size='11'>Mỗi hàng chọn mẫu nhiều mã có E cao nhất, hỗ trợ trong lớp ≥ 10%; màu theo thang 0–100%</text>")
        for class_id in 1:10
            x = left+(class_id-1)*cell_width+cell_width/2
            println(io, "<text x='$x' y='$(top-8)' text-anchor='middle' font-family='sans-serif' font-size='10'>$(svg_escape(FASHION_CLASSES[class_id]))</text>")
        end
        for (row_number, heat_row) in enumerate(heatmap_rows)
            index = heat_row.pattern_index
            y = top+(row_number-1)*cell_height
            label = "{"*join(all_patterns[index],",")*"} ("*string(FASHION_CLASSES[heat_row.target_class])*")"
            println(io, "<text x='$(left-8)' y='$(y+22)' text-anchor='end' font-family='sans-serif' font-size='11'>$(svg_escape(label))</text>")
            for class_id in 1:10
                rate = 100*class_rate_matrix[index,class_id]
                x = left+(class_id-1)*cell_width
                fill = svg_color(rate)
                text_color = rate >= 54 ? "white" : "#242424"
                println(io, "<rect x='$x' y='$y' width='$cell_width' height='$cell_height' fill='$fill' stroke='white'/>")
                println(io, "<text x='$(x+cell_width/2)' y='$(y+22)' text-anchor='middle' font-family='sans-serif' font-size='10' fill='$text_color'>$(round(rate;digits=1))</text>")
            end
        end
        println(io, "</svg>")
    end

    threshold_table_rows = [
        (string(row.percent,"%"), row.min_count, row.itemset_count,
         ismissing(row.median_emax) ? "—" : round(Float64(row.median_emax);digits=3),
         row.enriched_count,
         ismissing(row.enriched_fraction) ? "—" : string(round(100Float64(row.enriched_fraction);digits=2),"%"))
        for row in enrichment_threshold_rows
    ]
    class_table_rows = [(FASHION_CLASSES[c], class_population[c], class_enriched_counts[c]) for c in 1:10]
    example_table_rows = []
    for index in selected_pattern_indexes
        class_id = argmax(@view class_enrichment_matrix[index,:])
        push!(example_table_rows, ("{"*join(all_patterns[index],",")*"}",
            all_pattern_supports[index], round(100overall_rate_vector[index];digits=3),
            FASHION_CLASSES[class_id], round(100class_rate_matrix[index,class_id];digits=2),
            round(class_enrichment_matrix[index,class_id];digits=3)))
    end

    
class_rate_headers = Tuple("class_rate_" * replace(string(FASHION_CLASSES[c]), r"[^A-Za-z0-9]+" => "_") for c in 1:10)
class_enrichment_headers = Tuple("enrichment_" * replace(string(FASHION_CLASSES[c]), r"[^A-Za-z0-9]+" => "_") for c in 1:10)
all_itemset_class_rows = []
for index in 1:n_patterns
    base = ("{" * join(all_patterns[index], ",") * "}",
        all_pattern_supports[index], round(overall_rate_vector[index]; digits=8),
        round(max_enrichment[index]; digits=8))
    rates = Tuple(round(class_rate_matrix[index, c]; digits=8) for c in 1:10)
    enrichments = Tuple(round(class_enrichment_matrix[index, c]; digits=8) for c in 1:10)
    push!(all_itemset_class_rows, (base..., rates..., enrichments...))
end
write_csv(joinpath(output_dir, "itemset_class_enrichment_5_percent.csv"),
    ("itemset", "support_count", "overall_support_fraction", "Emax",
     class_rate_headers..., class_enrichment_headers...), all_itemset_class_rows)
write_csv(joinpath(output_dir, "enrichment_by_threshold.csv"),
    ("minsup_percent", "min_support_count", "itemset_count", "median_Emax",
     "itemsets_Emax_at_least_2", "fraction_Emax_at_least_2"), enrichment_threshold_rows)
write_csv(joinpath(output_dir, "enriched_itemset_counts_by_class.csv"),
    ("class_name", "class_image_count", "itemsets_with_E_at_least_2"),
    [(FASHION_CLASSES[c], class_population[c], class_enriched_counts[c]) for c in 1:10])
write_csv(joinpath(output_dir, "representative_patterns.csv"),
    ("itemset", "support_count", "overall_support_percent", "highest_enrichment_class",
     "class_support_percent", "Emax"), example_table_rows)
heatmap_csv_rows = []
for heat_row in heatmap_rows
    index = heat_row.pattern_index
    rates = Tuple(round(100class_rate_matrix[index, c]; digits=5) for c in 1:10)
    push!(heatmap_csv_rows, (FASHION_CLASSES[heat_row.target_class],
        "{" * join(all_patterns[index], ",") * "}",
        all_pattern_supports[index],
        round(class_enrichment_matrix[index, heat_row.target_class]; digits=8),
        rates...))
end
write_csv(joinpath(output_dir, "class_pattern_heatmap_data.csv"),
    ("target_class", "itemset", "support_count", "target_class_enrichment",
     Tuple(string(FASHION_CLASSES[c]) * "_support_percent" for c in 1:10)...),
    heatmap_csv_rows)

class_enrichment_summary_path = joinpath(output_dir, "class_enrichment_summary.md")
    open(class_enrichment_summary_path, "w") do io
        println(io, "## Hệ số làm giàu lớn nhất theo ngưỡng")
        println(io)
        println(io, markdown_table(threshold_table_rows,
            ("Ngưỡng","Số hỗ trợ tối thiểu","Số tập mục","Trung vị Emax","Số tập Emax ≥ 2","Tỷ lệ")))
        println(io)
        println(io, "Emax là hệ số làm giàu lớn nhất trong mười lớp. Các tập mục lồng nhau khi giảm minsup; số mẫu có Emax ≥ 2 có thể tăng khi thêm những mẫu ít phổ biến hơn.")
        println(io)
        println(io, "## Số tập mục được làm giàu theo lớp")
        println(io)
        println(io, markdown_table(class_table_rows, ("Lớp","Số ảnh lớp","Số tập có E(X,c) ≥ 2")))
        println(io)
        println(io, "Một tập có thể đạt E ≥ 2 ở nhiều lớp nên các số đếm giữa các lớp không loại trừ nhau.")
        println(io)
        println(io, "## Bốn mẫu đại diện để truy vết")
        println(io)
        println(io, markdown_table(example_table_rows,
            ("Tập mục","Số ảnh hỗ trợ","Hỗ trợ toàn cục","Lớp có E cao nhất","Tỷ lệ trong lớp","Emax")))
        println(io)
        println(io, "Quy tắc chọn: tập đơn có hỗ trợ cao nhất, cặp có hỗ trợ cao nhất, tập từ ba mã trở lên có hỗ trợ cao nhất, rồi một mẫu nhiều mã có hỗ trợ dưới 10%. Ưu tiên Jaccard không quá 0,80 với các mẫu đã chọn.")
        println(io)
        println(io, "Nhãn lớp chỉ dùng sau mining để mô tả. Phân tích gồm cả 60.000 ảnh huấn luyện và 10.000 ảnh kiểm thử nên không phải đánh giá phân loại độc lập.")
    end
    println("Itemsets at 5%: ", n_patterns, "; class-support sum checks: PASS")
    println("Representative patterns: ", example_patterns)
    show_demo_markdown(class_enrichment_summary_path)
    display(Markdown.parse("Phân bố Emax; đường đứt là E=2, râu biểu đồ kéo từ min đến max."))
    show_demo_image(boxplot_path; mime="image/svg+xml", alt="Phân bố hệ số làm giàu")
    display(Markdown.parse("Số tập mục có E(X,c) ≥ 2 ở từng lớp; một tập có thể được đếm ở nhiều lớp."))
    show_demo_image(class_count_path; mime="image/svg+xml", alt="Số tập mục được làm giàu theo lớp")
    display(Markdown.parse("Mỗi hàng heatmap là mẫu nhiều mã có E cao nhất cho lớp mục tiêu, với ít nhất 10% ảnh lớp hỗ trợ mẫu."))
    show_demo_image(heatmap_path; mime="image/svg+xml", alt="Heatmap phân bố mẫu theo lớp")
else
    example_patterns = Tuple{Vararg{Int}}[]
    
class_rate_headers = Tuple("class_rate_" * replace(string(FASHION_CLASSES[c]), r"[^A-Za-z0-9]+" => "_") for c in 1:10)
class_enrichment_headers = Tuple("enrichment_" * replace(string(FASHION_CLASSES[c]), r"[^A-Za-z0-9]+" => "_") for c in 1:10)
all_itemset_class_rows = []
for index in 1:n_patterns
    base = ("{" * join(all_patterns[index], ",") * "}",
        all_pattern_supports[index], round(overall_rate_vector[index]; digits=8),
        round(max_enrichment[index]; digits=8))
    rates = Tuple(round(class_rate_matrix[index, c]; digits=8) for c in 1:10)
    enrichments = Tuple(round(class_enrichment_matrix[index, c]; digits=8) for c in 1:10)
    push!(all_itemset_class_rows, (base..., rates..., enrichments...))
end
write_csv(joinpath(output_dir, "itemset_class_enrichment_5_percent.csv"),
    ("itemset", "support_count", "overall_support_fraction", "Emax",
     class_rate_headers..., class_enrichment_headers...), all_itemset_class_rows)
write_csv(joinpath(output_dir, "enrichment_by_threshold.csv"),
    ("minsup_percent", "min_support_count", "itemset_count", "median_Emax",
     "itemsets_Emax_at_least_2", "fraction_Emax_at_least_2"), enrichment_threshold_rows)
write_csv(joinpath(output_dir, "enriched_itemset_counts_by_class.csv"),
    ("class_name", "class_image_count", "itemsets_with_E_at_least_2"),
    [(FASHION_CLASSES[c], class_population[c], class_enriched_counts[c]) for c in 1:10])
write_csv(joinpath(output_dir, "representative_patterns.csv"),
    ("itemset", "support_count", "overall_support_percent", "highest_enrichment_class",
     "class_support_percent", "Emax"), example_table_rows)
heatmap_csv_rows = []
for heat_row in heatmap_rows
    index = heat_row.pattern_index
    rates = Tuple(round(100class_rate_matrix[index, c]; digits=5) for c in 1:10)
    push!(heatmap_csv_rows, (FASHION_CLASSES[heat_row.target_class],
        "{" * join(all_patterns[index], ",") * "}",
        all_pattern_supports[index],
        round(class_enrichment_matrix[index, heat_row.target_class]; digits=8),
        rates...))
end
write_csv(joinpath(output_dir, "class_pattern_heatmap_data.csv"),
    ("target_class", "itemset", "support_count", "target_class_enrichment",
     Tuple(string(FASHION_CLASSES[c]) * "_support_percent" for c in 1:10)...),
    heatmap_csv_rows)

class_enrichment_summary_path = joinpath(output_dir, "class_enrichment_summary.md")
    write(class_enrichment_summary_path, "Không thể phân tích theo lớp vì chưa có kết quả Eclat hợp lệ ở ngưỡng 5%.")
    show_demo_markdown(class_enrichment_summary_path)
end


## 8. Truy vết một mẫu theo ảnh

Nếu hai backend khớp ở ngưỡng 5% và có mẫu, notebook chọn mẫu nhiều từ có hỗ trợ cao nhất (phá hòa theo thứ tự từ điển), đếm lại trực tiếp, rồi nối mẫu với ảnh, patch và tâm cụm. Phân bố lớp dùng metadata chỉ sau khi mẫu đã được chọn.


In [ ]:
has_valid_five_percent_pattern = !isnothing(five_percent_result) && !isempty(five_percent_result)
# Chọn mẫu theo quy tắc cố định sau khi đối chiếu đã đạt PASS.
sample_summary = "Ngưỡng 5% chưa có cặp kết quả hợp lệ; không chọn mẫu."
if has_valid_five_percent_pattern
    multiword = [pair for pair in five_percent_result if length(first(pair)) > 1]
    candidates = isempty(multiword) ? collect(five_percent_result) : multiword
    sort!(candidates; by=pair -> (-last(pair), first(pair)))
    pattern, reported_support = first(candidates)
    supporting_tids = [tid for tid in eachindex(transactions)
        if all(item -> item in transactions[tid], pattern)]
    length(supporting_tids) == reported_support ||
        error("Direct support recount does not match the Eclat result.")
    sum(length, (transactions[tid] for tid in supporting_tids)) >= 0 ||
        error("Invalid supporting transaction set.")

    class_counts = zeros(Int, 10)
    for tid in supporting_tids
        image_id = input.image_id[tid]
        haskey(class_for_image, image_id) || error("A supporting transaction has no class metadata.")
        class_counts[class_for_image[image_id] + 1] += 1
    end
    sum(class_counts) == reported_support || error("Class counts do not sum to the direct support count.")
    class_population = zeros(Int, 10)
    for label in values(class_for_image)
        class_population[label+1] += 1
    end
    overall_support = reported_support / n_transactions

    sample_tid = first(supporting_tids)
    sample_image_id = input.image_id[sample_tid]
    sample_id_split, sample_source_index = parse_image_id(sample_image_id)
    patch_assignments = Dict{Int,NamedTuple}()
    assignment_file = artifact_path("patch_assignments")
    open(assignment_file, "r") do io
        readline(io) == "image_id,patch_id,patch_row,patch_col,word_id" ||
            error("Unexpected patch assignment table header.")
        for line in eachline(io)
            fields = split(line, ',')
            fields[1] == sample_image_id || continue
            word_id = parse(Int, fields[5])
            word_id in pattern || continue
            patch_id = parse(Int, fields[2])
            row, column = parse(Int, fields[3]), parse(Int, fields[4])
            1 <= row <= 7 && 1 <= column <= 7 && 1 <= patch_id <= 49 ||
                error("Patch coordinates are outside the 7×7 grid.")
            patch_id == (row-1)*7 + column || error("Patch ID does not match its row and column.")
            get!(patch_assignments, word_id, (; patch_id, row, column))
            length(patch_assignments) == length(pattern) && break
        end
    end
    Set(keys(patch_assignments)) == Set(pattern) ||
        error("Could not trace every word in the selected pattern to a patch.")
end

### Tải ảnh gốc và dựng vùng ảnh

Tải hoặc kiểm tra lại Fashion-MNIST theo checksum, lấy ảnh hỗ trợ mẫu đã chọn và lưu ảnh gốc, các patch cùng tâm cụm tương ứng.

In [ ]:
if has_valid_five_percent_pattern
    dataset_path = joinpath(bundle_root, split(demo["dataset_manifest"]["path"], '/')...)
    dataset_manifest = read_dataset_manifest(dataset_path)
    data_dir = joinpath(repo_root, "data", "raw", "fashion-mnist")
    download_fashion_mnist(data_dir)
    for (filename, expected_hash) in dataset_manifest["archive_sha256"]
        actual_hash = bytes2hex(open(sha256, joinpath(data_dir, filename)))
        actual_hash == lowercase(String(expected_hash)) || error("Official source archive hash mismatch: $filename")
    end
    # Recheck manifest paths against the verified copies just downloaded into this repository checkout.
    read_dataset_manifest(dataset_path; verify_archives=true, repo_root)
    split_data = load_fashion_mnist(data_dir; split=sample_id_split)
    image = split_data.images[:, :, sample_source_index + 1]
    PNGFiles.save(joinpath(output_dir, "supporting_image.png"), repeat(image; inner=(8, 8)))
    centroids = zeros(Float32, 16, 32)
    centroid_seen = falses(16, 32)
    open(artifact_path("centroids"), "r") do io
        readline(io) == "word_id,pixel_index,centroid_value" || error("Unexpected centroid table header.")
        for line in eachline(io)
            fields = split(line, ',')
            length(fields) == 3 || error("Malformed centroid row.")
            word_id, pixel_index = parse(Int, fields[1]), parse(Int, fields[2])
            1 <= word_id <= 32 && 1 <= pixel_index <= 16 || error("Centroid index outside the codebook.")
            centroid_seen[pixel_index, word_id] && error("Duplicate centroid coordinate.")
            centroid_seen[pixel_index, word_id] = true
            centroids[pixel_index, word_id] = parse(Float32, fields[3])
        end
    end
    all(centroid_seen) || error("Centroid table is incomplete.")
    for word_id in pattern
        patch_info = patch_assignments[word_id]
        y0 = (patch_info.row-1)*4 + 1
        x0 = (patch_info.column-1)*4 + 1
        actual_patch = image[y0:y0+3, x0:x0+3]
        PNGFiles.save(joinpath(output_dir, "word_$(word_id)_actual.png"),
            repeat(actual_patch; inner=(16, 16)))
        prototype = reshape(centroids[:, word_id], 4, 4)
        prototype_pixels = round.(UInt8, clamp.(prototype, 0, 1) .* 255)
        PNGFiles.save(joinpath(output_dir, "word_$(word_id)_centroid.png"),
            repeat(prototype_pixels; inner=(16, 16)))
    end
end

### Phân tích mẫu theo lớp

Đếm ảnh hỗ trợ theo nhãn lớp, tính tỷ lệ trong từng lớp và hệ số làm giàu, rồi ghi báo cáo truy vết mẫu.

In [ ]:
if has_valid_five_percent_pattern
    class_rows = []
    for label in 0:9
        count = class_counts[label+1]
        class_rate = count / class_population[label+1]
        enrichment = overall_support == 0 ? 0.0 : class_rate / overall_support
        push!(class_rows, (label, FASHION_CLASSES[label+1], count,
            class_population[label+1], round(100class_rate; digits=2), round(enrichment; digits=3)))
    end

    write_csv(joinpath(output_dir, "primary_pattern_class_distribution.csv"),
        ("class_label", "class_name", "supporting_images", "class_population",
         "class_support_percent", "enrichment"), class_rows)
    pattern_trace_rows = [
        ("{" * join(pattern, ",") * "}", reported_support, round(overall_support; digits=8),
         sample_image_id, input.tid[sample_tid], word_id, patch_assignments[word_id].patch_id,
         patch_assignments[word_id].row, patch_assignments[word_id].column)
        for word_id in pattern
    ]
    write_csv(joinpath(output_dir, "primary_pattern_patch_trace.csv"),
        ("itemset", "support_count", "overall_support_fraction", "example_image_id",
         "tid", "visual_word_id", "patch_id", "patch_row", "patch_column"), pattern_trace_rows)

    io = IOBuffer()
    println(io, "## Mẫu được truy vết")
    println(io)
    println(io, "- Mẫu từ thị giác: [", join(pattern, ", "), "].")
    println(io, "- Ngưỡng: 5%; số hỗ trợ tối thiểu: ", cld(5*n_transactions,100), ".")
    println(io, "- Số hỗ trợ Eclat: ", reported_support, "; đếm lại trực tiếp: ", length(supporting_tids), ".")
    println(io, "- Tỷ lệ hỗ trợ toàn bộ tập ảnh: ", round(100overall_support; digits=3), "%.")
    println(io, "- Ảnh minh họa: ", sample_image_id, "; TID: ", input.tid[sample_tid],
        "; số lớp theo metadata: ", class_for_image[sample_image_id], ".")
    println(io, "- Các vùng ảnh được truy vết:")
    for word_id in pattern
        info = patch_assignments[word_id]
        println(io, "  - Từ thị giác ", word_id, ": vùng ", info.patch_id,
            ", hàng ", info.row, ", cột ", info.column, ".")
    end
    println(io)
    println(io, "### Phân bố số ảnh hỗ trợ theo lớp")
    println(io)
    println(io, "| Nhãn | Lớp | Ảnh hỗ trợ | Tổng ảnh lớp | Tỷ lệ trong lớp | Hệ số làm giàu |")
    println(io, "|---:|---|---:|---:|---:|---:|")
    for row in class_rows
        println(io, "| ", join(string.(row), " | "), " |")
    end
    println(io)
    println(io, "Hệ số làm giàu là tỷ lệ chứa mẫu trong lớp chia cho tỷ lệ chứa mẫu trong toàn bộ tập quan sát. Kết quả mô tả đồng xuất hiện và phân bố theo lớp; không phải kết quả phân loại.")
    ranked_by_enrichment = sort(class_rows; by=row -> row[6], rev=true)
    highest_enrichment = first(ranked_by_enrichment)
    lowest_enrichment = last(ranked_by_enrichment)
    println(io)
    println(io, "### Diễn giải phân bố theo lớp")
    println(io)
    println(io, "- Tỷ lệ chứa mẫu cao nhất là ", highest_enrichment[5], "% ở lớp ", highest_enrichment[2],
        " (E=", highest_enrichment[6], "); thấp nhất là ", lowest_enrichment[5], "% ở lớp ",
        lowest_enrichment[2], " (E=", lowest_enrichment[6], ").")
    println(io, "- Mẫu xuất hiện ở cả mười lớp. Chênh lệch tỷ lệ mô tả mức độ phổ biến khác nhau giữa lớp; đây không phải dự đoán nhãn hay bằng chứng về quan hệ nhân quả.")
    println(io, "- Trong ảnh minh họa ", sample_image_id, ", các vùng được truy vết có tọa độ đã liệt kê ở trên. Một ảnh ví dụ không xác lập vị trí cố định cho mẫu; BoVW nhị phân chỉ ghi nhận sự hiện diện của các mã trong ảnh.")
    sample_summary = String(take!(io))
    open(joinpath(output_dir, "pattern_summary.md"), "w") do file
        write(file, sample_summary)
    end
else
    open(joinpath(output_dir, "pattern_summary.md"), "w") do file
        write(file, sample_summary)
    end
end

### Minh họa bốn mẫu đại diện

Với mỗi mẫu, notebook chọn ảnh hỗ trợ thuộc lớp có hệ số làm giàu cao nhất, rồi truy vết từng mã về một vùng ảnh thực tế và tâm cụm tương ứng. Ảnh chỉ minh họa một trường hợp gán mã; không khẳng định vị trí cố định của mẫu.


In [ ]:
if !isempty(example_patterns) && has_valid_five_percent_pattern
    example_visual_dir = joinpath(output_dir, "pattern_examples")
    mkpath(example_visual_dir)
    desired_words_by_image = Dict{String,Set{Int}}()
    example_tids = Int[]
    example_image_ids = String[]
    example_target_classes = Int[]

    for itemset in example_patterns
        pattern_index = pattern_index_by_tuple[itemset]
        target_class = argmax(@view class_enrichment_matrix[pattern_index,:])
        target_label = target_class - 1
        tid = findfirst(eachindex(transactions)) do candidate_tid
            class_for_image[input.image_id[candidate_tid]] == target_label &&
            all(item -> item in transactions[candidate_tid], itemset)
        end
        isnothing(tid) && error("No target-class supporting transaction for $itemset.")
        image_id = input.image_id[tid]
        push!(example_tids, tid)
        push!(example_image_ids, image_id)
        push!(example_target_classes, target_class)
        union!(get!(desired_words_by_image, image_id, Set{Int}()), itemset)
    end

    example_patches = Dict{String,Dict{Int,NamedTuple}}()
    open(artifact_path("patch_assignments"), "r") do io
        readline(io) == "image_id,patch_id,patch_row,patch_col,word_id" ||
            error("Unexpected patch assignment table header.")
        for line in eachline(io)
            fields = split(line, ',')
            image_id = fields[1]
            haskey(desired_words_by_image, image_id) || continue
            word_id = parse(Int, fields[5])
            word_id in desired_words_by_image[image_id] || continue
            patch_id, row, column = parse(Int, fields[2]), parse(Int, fields[3]), parse(Int, fields[4])
            patch_id == (row-1)*7 + column || error("Patch ID does not match its row and column.")
            get!(example_patches, image_id, Dict{Int,NamedTuple}())
            get!(example_patches[image_id], word_id, (; patch_id, row, column))
        end
    end

    split_cache = Dict{Symbol,Any}()
    for (example_number, itemset) in enumerate(example_patterns)
        tid, image_id = example_tids[example_number], example_image_ids[example_number]
        split_name, source_index = parse_image_id(image_id)
        split_data = get!(split_cache, split_name) do
            load_fashion_mnist(data_dir; split=split_name)
        end
        image = split_data.images[:,:,source_index+1]
        image_path = joinpath(example_visual_dir, "example_$(example_number)_image.png")
        PNGFiles.save(image_path, repeat(image; inner=(8,8)))
        isfile(image_path) || error("Failed to save example image: $image_path")
        # Multiple example patterns can share an image; its patch map contains their union.
        image_patches = get(example_patches, image_id, Dict{Int,NamedTuple}())
        missing_words = setdiff(Set(itemset), Set(keys(image_patches)))
        isempty(missing_words) || error("Visual trace is missing patch assignments for $itemset in $image_id; missing words: $(sort!(collect(missing_words))).")
        println("Pattern ", itemset, " | support=", five_percent_result[itemset],
            " | image=", image_id, " | TID=", input.tid[tid],
            " | target class=", FASHION_CLASSES[example_target_classes[example_number]])
        for word_id in itemset
            info = image_patches[word_id]
            y0, x0 = (info.row-1)*4+1, (info.column-1)*4+1
            patch = image[y0:y0+3, x0:x0+3]
            actual_path = joinpath(example_visual_dir, "example_$(example_number)_word_$(word_id)_actual.png")
            centroid_path = joinpath(example_visual_dir, "example_$(example_number)_word_$(word_id)_centroid.png")
            PNGFiles.save(actual_path, repeat(patch; inner=(16,16)))
            isfile(actual_path) || error("Failed to save actual patch: $actual_path")
            prototype = reshape(centroids[:,word_id], 4, 4)
            prototype_pixels = round.(UInt8, clamp.(prototype,0,1) .* 255)
            PNGFiles.save(centroid_path, repeat(prototype_pixels; inner=(16,16)))
            isfile(centroid_path) || error("Failed to save centroid image: $centroid_path")
        end
    end
    println("Example images saved under: ", example_visual_dir)
else
    display(Markdown.parse("Không có mẫu hợp lệ ở ngưỡng 5% để dựng ảnh đại diện."))
end


### Gộp báo cáo phân tích

Ghép báo cáo khai thác với phần truy vết mẫu thành một tệp tóm tắt cho lượt chạy.

In [ ]:
open(joinpath(output_dir, "analysis_summary.md"), "w") do io
    write(io, read(joinpath(output_dir, "mining_summary.md"), String))
    println(io)
    write(io, read(joinpath(output_dir, "class_enrichment_summary.md"), String))
    println(io)
    write(io, sample_summary)
end
println(joinpath(output_dir, "analysis_summary.md"))


### Hiển thị kết quả truy vết

Cell này đọc các ảnh do cell “Minh họa bốn mẫu đại diện” tạo trong cùng thư mục kết quả. Nếu thiếu file, hãy chạy cell tạo ảnh trước rồi chạy lại cell này.


In [ ]:
show_demo_markdown(joinpath(ANALYSIS_DIR, "pattern_summary.md"))
if !isempty(example_patterns) && isdir(joinpath(ANALYSIS_DIR, "pattern_examples"))
    missing_example_files = String[]
    for example_number in eachindex(example_patterns)
        itemset = example_patterns[example_number]
        push!(missing_example_files,
            joinpath(ANALYSIS_DIR, "pattern_examples", "example_$(example_number)_image.png"))
        for word_id in itemset
            push!(missing_example_files,
                joinpath(ANALYSIS_DIR, "pattern_examples", "example_$(example_number)_word_$(word_id)_actual.png"))
            push!(missing_example_files,
                joinpath(ANALYSIS_DIR, "pattern_examples", "example_$(example_number)_word_$(word_id)_centroid.png"))
        end
    end
    missing_example_files = filter(path -> !isfile(path), missing_example_files)
    if isempty(missing_example_files)
        display(Markdown.parse("Bốn mẫu đại diện, ảnh hỗ trợ, các vùng được truy vết và tâm cụm tương ứng:"))
        for example_number in eachindex(example_patterns)
            itemset = example_patterns[example_number]
            image_path = joinpath(ANALYSIS_DIR, "pattern_examples", "example_$(example_number)_image.png")
            display(Markdown.parse("Mẫu {$(join(itemset, ", "))}."))
            show_demo_image(image_path; alt="Ảnh hỗ trợ của mẫu $(join(itemset, ", "))")
            for word_id in itemset
                actual_path = joinpath(ANALYSIS_DIR, "pattern_examples", "example_$(example_number)_word_$(word_id)_actual.png")
                centroid_path = joinpath(ANALYSIS_DIR, "pattern_examples", "example_$(example_number)_word_$(word_id)_centroid.png")
                show_demo_image(actual_path; alt="Vùng ảnh của từ thị giác $word_id")
                show_demo_image(centroid_path; alt="Tâm cụm của từ thị giác $word_id")
            end
        end
    else
        display(Markdown.parse("Chưa có đủ ảnh patch trong thư mục kết quả hiện tại. Hãy chạy lại cell ‘Minh họa bốn mẫu đại diện’ ở phía trên, rồi chạy lại cell hiển thị này."))
        println("Thư mục kết quả hiện tại: ", ANALYSIS_DIR)
        println("Các file chưa được tạo:")
        println.(missing_example_files)
    end
else
    display(Markdown.parse("Không có mẫu hợp lệ ở ngưỡng 5% để dựng ảnh đại diện."))
end


## 9. Lưu và tải kết quả

Notebook lưu các bảng, biểu đồ, ảnh truy vết, kết quả benchmark, manifest và gói đầu vào trong một thư mục duy nhất. Cell kế tiếp tạo file nén để tải về. Trên Colab, thư mục mặc định nằm trong /content; mở Files panel, vào thư mục fashion-demo-results-... rồi tải fashion-demo-analysis.tar.gz. Vì /content thuộc runtime tạm thời, hãy tải file về trước khi ngắt hoặc khởi động lại runtime.

Có thể đặt FASHION_DEMO_RESULTS_ROOT trước khi chạy để chọn thư mục gốc khác, chẳng hạn vị trí Drive đã mount.


In [ ]:
archive_name = "fashion-demo-analysis.tar.gz"
archive_path = joinpath(ANALYSIS_DIR, archive_name)
archive_staging_path = joinpath(dirname(ANALYSIS_DIR), basename(ANALYSIS_DIR) * ".tar.gz")

readme_path = joinpath(ANALYSIS_DIR, "README.md")
open(readme_path, "w") do io
    println(io, "# Fashion-MNIST Notebook 3 results")
    println(io)
    println(io, "- Run ID: ", demo["run_id"])
    println(io, "- Transaction SHA-256: ", input.input_sha256)
    println(io, "- Source commit: ", demo["source_commit"])
    println(io, "- Result folder: ", ANALYSIS_DIR)
    println(io)
    println(io, "## Main files")
    println(io)
    println(io, "- analysis_summary.md: combined numerical summary and interpretation")
    println(io, "- CSV files: transaction, mining, benchmark, itemset-class enrichment, and trace tables")
    println(io, "- SVG files: transaction lengths, word frequencies, enrichment distribution, class counts, and heatmap")
    println(io, "- PNG files: example images, assigned patches, and centroids")
    println(io, "- eclat_measurements/: worker itemset JSONL, metrics TOML, and logs by backend and minsup")
    println(io, "- input_bundle.tar.gz and copied manifests: verified source artifacts")
    println(io)
    println(io, "The archive contains the result files created before packaging; it does not contain itself.")
end

open(archive_staging_path, "w") do raw_io
    gzip_io = GzipCompressorStream(raw_io)
    try
        Tar.create(ANALYSIS_DIR, gzip_io; portable=true)
    finally
        close(gzip_io)
    end
end
mv(archive_staging_path, archive_path; force=true)

println("Results folder: ", ANALYSIS_DIR)
println("Downloadable archive: ", archive_path)
println("The folder contains the archive, CSV/Markdown tables, charts, images, worker results, manifests, and source bundle.")


## 10. Kết luận và giới hạn diễn giải

Các tập mục mô tả những từ thị giác cùng xuất hiện trong giao dịch. Phân bố lớp và hệ số làm giàu chỉ mô tả nhóm ảnh chứa mẫu; đây không phải kết quả phân loại hay bằng chứng nhân quả. Codebook được học từ cả hai split trong pipeline này, vì vậy kết quả theo lớp cần được trình bày như phân tích khai phá trên dữ liệu đã xử lý.

Thời gian và peak RSS được đo trong tiến trình Julia riêng cho từng backend/ngưỡng, nhưng đây là phép đo đơn không warm-up; RSS gồm cả Julia và bước nạp giao dịch. Khi so sánh giữa máy hoặc lần chạy, cần đối chiếu phiên bản Julia, CPU và trạng thái cache.
